# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library. All references to dataset entities—record sets, fields, and columns—are made using their Croissant `@id`.

### Dataset Source
The dataset is accessed via the following Croissant schema URL:

[https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json](https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json)

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Name:", metadata.name)
print("Description:", metadata.description)
print("Published:", getattr(metadata, 'datePublished', None))
print("License:", getattr(metadata, 'license', None))

## 2. Data Overview
Review available record sets, their associated fields and columns, and their Croissant `@id`s for referencing.

We'll enumerate all record sets (tables) and their fields, printing the `@id` and labels for each.

In [ ]:
# Display record sets, fields, and columns with their @id
record_sets = dataset.record_sets()

if not record_sets:
    print("No record sets detected in metadata. If you see this, please check the Croissant schema.")
else:
    overview = []
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        print(f"  Name: {rs.get('name', 'N/A')}")
        if 'field' in rs:
            fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
            for fld in fields:
                print(f"    Field @id: {fld['@id']}, Label: {fld.get('name', 'N/A')}")
                if 'column' in fld:
                    columns = fld['column'] if isinstance(fld['column'], list) else [fld['column']]
                    for col in columns:
                        print(f"      Column @id: {col['@id']}, Label: {col.get('name', 'N/A')}")
        print('-'*60)
        overview.append(rs['@id'])

## 3. Data Extraction
Load data from each record set (`@id`) into a DataFrame for analysis using `mlcroissant`. Use the `@id` from the overview.

In [ ]:
# Extract all record sets to pandas DataFrames
dataframes = {}

if not record_sets:
    print("No record sets found.")
else:
    for rs in record_sets:
        rs_id = rs['@id']
        print(f"Loading records for record set @id: {rs_id}")
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Columns in record set {rs_id}:", df.columns.tolist())
            print(df.head(2))
        else:
            print(f"No records found for record set {rs_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps: filtering, normalizing, and grouping. We reference fields using their `@id`.

Below, select a record set and a numeric field by their `@id` for demonstration. If your metadata overview printed field/column IDs, replace them as needed.

In [ ]:
# Choose a record set and field (update if your overview shows different IDs)
if not dataframes:
    print("No DataFrames available. Cannot perform EDA.")
else:
    # Pick one record set @id in this example
    chosen_rs_id = list(dataframes.keys())[0]
    df = dataframes[chosen_rs_id]

    print(f"Performing EDA on record set: {chosen_rs_id}")
    
    # Try to identify a numeric column
    numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
    if numeric_columns:
        numeric_field_id = numeric_columns[0]  # Use column name as field @id
        print(f"Using numeric field: {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if not df[numeric_field_id].isnull().all() else 1
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold}:")
        print(filtered_df.head(2))

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print("Normalized values:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].head(3)])

        # Grouping by another field if available
        potential_groups = [col for col in df.columns if col != numeric_field_id]
        if potential_groups:
            group_field_id = potential_groups[0]
            print(f"Grouping filtered records by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(grouped_df.head(3))
        else:
            print("No non-numeric field found to group by.")
    else:
        print("No numeric fields found in the record set.")

## 5. Visualization
Visualize the distribution of a numeric field from the dataset, using `matplotlib` and `seaborn` if available.

Replace field names with their `@id` if appropriate:

In [ ]:
# Visualization of numeric field distribution
import matplotlib.pyplot as plt
import seaborn as sns

if not dataframes:
    print("No DataFrames available for plotting.")
else:
    df_vis = dataframes[chosen_rs_id]
    if numeric_columns:
        plt.figure(figsize=(8, 5))
        sns.histplot(df_vis[numeric_field_id], bins=6, kde=True)
        plt.title(f"Distribution of {numeric_field_id} (Record set: {chosen_rs_id})")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Frequency")
        plt.tight_layout()
        plt.show()
    else:
        print("No numeric columns to visualize.")

## 6. Conclusion
We demonstrated how to explore the Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management dataset using `mlcroissant`. Fields, columns, and tables were referenced via `@id`, complying with FAIR² best practices.

Key findings:
- Dataset loaded and record sets identified by `@id`.
- Fields and columns inspected and extracted as DataFrames.
- Standard data processing with filtering, normalization, and grouping performed.
- Basic visualization illustrated for numeric column distributions.

This workflow can be extended for further clinical or research analysis using field and record set references by `@id`.